# Tokenization Homework: Llama 2 vs Llama 3

In this assignment, we will investigate the differences in how English and Persian texts are tokenized by two popular language models: Llama 2 and Llama 3. 

**Important Notes for this Assignment:**
1. **No GPU Required:** We are ONLY loading the tokenizers, not the massive model weights. Tokenizers are very small files (just a few megabytes) and run perfectly fast on any standard laptop CPU.
2. **Small Download Size:** By using `AutoTokenizer.from_pretrained()`, the code will *only* download the tokenizer configuration files (`tokenizer.json`, `tokenizer.model`, etc.). It will **NOT** download the gigabytes of `.safetensors` or `.bin` model weights. The total download will be less than 5 MB.
3. **Local Downloads:** We will configure the environment to download the tokenizer files using `devneeds.ir` to avoid network restrictions. We also use a local PyPI mirror (Runflare) for installing packages.

---

### Setup
Run the cell below to set up your environment.


In [ ]:
# Install necessary libraries using an Iranian PyPI mirror (devneeds.ir) to bypass network issues
%pip install -i https://pypi.devneeds.ir/simple/ transformers sentencepiece tiktoken


In [ ]:
import os

# Set environment variable to use the devneeds.ir mirror for HuggingFace
os.environ["HF_ENDPOINT"] = "https://hf.devneeds.ir"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "300"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import transformers
from transformers import AutoTokenizer

print(f"Transformers version: {transformers.__version__}")


## Step 1: Loading Tokenizers

In this section, you need to load the tokenizers for Llama 2 and Llama 3.

- Llama 2 Tokenizer: `meta-llama/Llama-2-7b-hf`
- Llama 3 Tokenizer: `meta-llama/Meta-Llama-3-8B-Instruct`


In [ ]:
# TODO: Load Llama 2 Tokenizer
tokenizer_llama2 = ...

# TODO: Load Llama 3 Tokenizer
tokenizer_llama3 = ...


## Step 2: Sample Texts

Here are two sample texts (one in English and one in Persian). You can replace them with your own texts if you prefer.


In [ ]:
text_en = "Artificial intelligence is a rapidly growing field that aims to create machines capable of intelligent behavior."

text_fa = "هوش مصنوعی حوزه‌ای به سرعت در حال رشد است که هدف آن ساخت ماشین‌هایی با قابلیت رفتار هوشمندانه است."

print("English Text:", text_en)
print("Persian Text:", text_fa)


## Step 3: Tokenizing Texts

Now, tokenize both the English and Persian texts using both tokenizers. The output should be a list of token IDs.


In [ ]:
# TODO: Tokenize texts using Llama 2 tokenizer
tokens_en_llama2 = ...
tokens_fa_llama2 = ...

print(f"Llama 2 (EN) tokens count: {len(tokens_en_llama2) if tokens_en_llama2 is not Ellipsis else 'Not implemented'}")
print(f"Llama 2 (FA) tokens count: {len(tokens_fa_llama2) if tokens_fa_llama2 is not Ellipsis else 'Not implemented'}")


In [ ]:
# TODO: Tokenize texts using Llama 3 tokenizer
tokens_en_llama3 = ...
tokens_fa_llama3 = ...

print(f"Llama 3 (EN) tokens count: {len(tokens_en_llama3) if tokens_en_llama3 is not Ellipsis else 'Not implemented'}")
print(f"Llama 3 (FA) tokens count: {len(tokens_fa_llama3) if tokens_fa_llama3 is not Ellipsis else 'Not implemented'}")


## Step 4: Average Characters per Token

To better understand how these tokenizers behave, let's calculate the average number of original text characters covered by a single token.
A more efficient tokenizer for a specific language will group more characters into a single token (resulting in a higher average length). Conversely, if the tokenizer is unfamiliar with the language, it will fall back to splitting words into very small chunks or individual characters (resulting in a lower average length).

**Formula:** (Number of characters in the original text) / (Number of tokens)


In [ ]:
# TODO: Calculate average characters per token for Llama 2
# 1. English
avg_len_en_llama2 = ...

# 2. Persian
avg_len_fa_llama2 = ...

print("--- Llama 2 ---")
print(f"Average char per token (EN): {avg_len_en_llama2}")
print(f"Average char per token (FA): {avg_len_fa_llama2}")


In [ ]:
# TODO: Calculate average characters per token for Llama 3
# 1. English
avg_len_en_llama3 = ...

# 2. Persian
avg_len_fa_llama3 = ...

print("--- Llama 3 ---")
print(f"Average char per token (EN): {avg_len_en_llama3}")
print(f"Average char per token (FA): {avg_len_fa_llama3}")


## Step 5: Analysis and Conclusion

Based on the numbers you calculated in the previous sections, answer the following questions briefly:

**Question 1:** What is the difference in average token length between English and Persian when using the Llama 2 tokenizer? What does this indicate?
**Answer:** (Write your answer here)

**Question 2:** How does Llama 3's performance on Persian text compare to Llama 2? (Refer to the number of tokens and their average length).
**Answer:** (Write your answer here)

**Question 3:** (Bonus) What is the main reason for this improvement in Llama 3? (Hint: search the internet for the vocabulary size of Llama 2 vs Llama 3)
**Answer:** (Write your answer here)


# Human Preference Alignment

### Setup
Run the cell below to set up your environment.

In [ ]:
import subprocess, sys

packages = [
    "transformers>=4.45.0",
    "datasets>=2.18.0",
    "trl>=0.9.0",
    "torch>=2.0.0",
    "accelerate>=0.28.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.12.0",
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "evaluate",
    "tqdm",
    "einops",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed successfully!")

## Cell 0 — Imports, Config & Global Settings

Set seeds, define device (`CUDA` / `CPU`), precision (`bfloat16` / `float32`), and all hyperparameters in one place. A shared `COLORS` dict and `plt.rcParams` keep all plots visually consistent.

> **All hyperparameters live here** — change `STEPS_*`, `LR_*`, `MAX_LENGTH`, or `N_TRAIN` before running any training cell.

In [ ]:
import os, warnings, random, json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
)
from datasets import load_dataset, Dataset

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

set_seed(42)
random.seed(42)
np.random.seed(42)


#CPU
DEVICE = torch.device("cpu")
DTYPE = torch.float32
USE_AMP = False

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LENGTH = 128

N_TRAIN = #
N_EVAL  #

# ── Learning rates (each stage needs its own LR) ────────
LR_SFT  = # YOUR CODE HERE
LR_RM   = # YOUR CODE HERE
LR_PPO  = # YOUR CODE HERE
LR_DPO  = # YOUR CODE HERE
LR_ORPO = # YOUR CODE HERE

# ── Alignment hyperparameters ────────────────────────────
BETA_DPO    = # YOUR CODE HERE  # KL regularization strength in DPO
LAMBDA_ORPO = # YOUR CODE HERE  # weight of odds-ratio term in ORPO
KL_COEF     = # YOUR CODE HERE  # KL penalty coefficient in PPO

# ── Training steps per stage ─────────────────────────────
STEPS_SFT  = # YOUR CODE HERE
STEPS_RM   = # YOUR CODE HERE
STEPS_PPO  = # YOUR CODE HERE
STEPS_DPO  = # YOUR CODE HERE
STEPS_ORPO = # YOUR CODE HERE

BATCH_SIZE = # YOUR CODE HERE

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f9fa",
    "axes.grid":        True,
    "grid.alpha":       0.4,
    "font.size":        11,
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "lines.linewidth":  2.0,
})
COLORS = {
    "SFT":      "#3498db",
    "RM":       "#e67e22",
    "PPO":      "#e74c3c",
    "DPO":      "#9b59b6",
    "ORPO":     "#1abc9c",
    "chosen":   "#2ecc71",
    "rejected": "#e74c3c",
}

print("Configuration summary:")
print(f" Device   : {DEVICE} (CPU)")
print(f" Precision: {DTYPE}")
print(f" AMP enabled : {USE_AMP}")

if DEVICE.type == "cuda":
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU VRAM  : {mem:.1f} GB")
    
print(f" Model : {MODEL_NAME}")
print(f" Max Length: {MAX_LENGTH} tokens")
print(f" N_TRAIN  : {N_TRAIN}  |  N_EVAL: {N_EVAL}")
print(f" Batch size: {BATCH_SIZE}")

## Cell 1 — Dataset: Anthropic HH-RLHF

Load `Anthropic/hh-rlhf` from HuggingFace — a real-world preference dataset of **chosen vs rejected** conversation pairs used to train harmless, helpful assistants.

> Each sample contains a multi-turn conversation ending with two responses: `chosen` (preferred) and `rejected` (dispreferred).

Note: If you do not have internet access, use the local Excel file dataset.xlsx instead.

In [ ]:
# Load hh-rlhf dataset — specify the correct split names
raw_train = load_dataset(...)
raw_test  = load_dataset(...)

print(f" Train samples : {len(raw_train):,}")
print(f" Test  samples : {len(raw_test):,}")
print(f" Columns       : {raw_train.column_names}")

# Print one example — access the 'chosen' and 'rejected' fields
ex = raw_train[1]
print(f"\n[CHOSEN]:\n{ex[...]}...")
print(f"\n[REJECTED]:\n{ex[...]}...")

## Cell 2 — EDA: Chosen vs Rejected Response Analysis

Analyze length distributions across **800 samples** with 6 plots:
- **Histograms** of chosen vs rejected token lengths
- **Boxplots** for spread and outliers
- **CDF** to compare cumulative distributions
- **Scatter** plot of chosen vs rejected lengths per sample
- **Descriptive stats** table (mean, std, median, min, max)

> Longer ≠ better — but length asymmetry between chosen and rejected is a useful signal before training.

In [ ]:
EDA_SIZE = 800
eda_sub  = raw_train.select(range(EDA_SIZE))

# Compute word counts for chosen and rejected responses
chosen_lens   = [...]
rejected_lens = [...]

# Compute length difference (chosen - rejected) per sample
len_diff = [...]

# Extract prompt length by splitting on "\n\nAssistant:"
def get_prompt_len(text):
    # YOUR CODE HERE
    pass

prompt_lens = [get_prompt_len(ex["chosen"]) for ex in eda_sub]

# ── Plotting (6 subplots in a 2×3 grid) ──────────────────
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, ...)

# Plot 1 — Histogram: chosen vs rejected lengths
ax1 = fig.add_subplot(gs[0, 0])
# YOUR CODE HERE

# Plot 2 — Boxplot comparison
ax2 = fig.add_subplot(gs[0, 1])
# YOUR CODE HERE

# Plot 3 — Histogram of length differences
ax3 = fig.add_subplot(gs[0, 2])
# YOUR CODE HERE

# Plot 4 — CDF of chosen vs rejected
ax4 = fig.add_subplot(gs[1, 0])
# YOUR CODE HERE

# Plot 5 — Scatter: prompt length vs response length
ax5 = fig.add_subplot(gs[1, 1])
# YOUR CODE HERE

# Plot 6 — Bar chart of descriptive statistics (mean, median, std)
ax6 = fig.add_subplot(gs[1, 2])
# YOUR CODE HERE

plt.savefig("eda_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 3 — Tokenizer & Preprocessing

Load the tokenizer and define ChatML formatters (`make_chatml_prompt_only`, `make_chatml_full`). Then parse HH-RLHF into two DataFrames:
- **`sft_df`** — prompt + chosen response (for SFT)
- **`pref_df`** — prompt + chosen + rejected pairs (for RM / DPO / ORPO)

In [ ]:
# Load tokenizer — set pad_token if missing, padding_side = "right"
tokenizer = AutoTokenizer.from_pretrained(...)
if tokenizer.pad_token is None:
    # YOUR CODE HERE
    pass

SYSTEM_PROMPT = "You are a helpful, harmless, and honest assistant."

def parse_hh_rlhf(text, max_chars=350):
    # Split on "\n\nAssistant:" to separate prompt from response
    # Extract the last Human turn from the prompt
    # Return (prompt, response) — empty string if not found
    pass

def make_chatml_text(prompt, response):
    # Build a 3-turn message list: system / user / assistant
    # Apply chat template — tokenize=False
    pass

def make_chatml_prompt_only(prompt):
    # Build a 2-turn message list: system / user
    # Apply chat template — add_generation_prompt=True
    pass

# ── Preprocessing loop ────────────────────────────────────
sft_rows, pref_rows = [], []

for i in tqdm(range(N_TRAIN + N_EVAL), desc="Preprocessing"):
    ex = raw_train[i]

    # Parse chosen and rejected texts into (prompt, response)
    p_ch, r_ch = parse_hh_rlhf(...)
    p_rj, r_rj = parse_hh_rlhf(...)

    # Append to sft_rows: needs 'prompt' and 'completion'
    if p_ch and r_ch:
        sft_rows.append({...})

    # Append to pref_rows: needs prompt, chosen, rejected + chatml versions
    if p_ch and r_ch and r_rj:
        pref_rows.append({...})

# ── Build DataFrames and split train/eval ─────────────────
sft_df  = pd.DataFrame(sft_rows).reset_index(drop=True)
pref_df = pd.DataFrame(pref_rows).reset_index(drop=True)

sft_train_df  = ...
sft_eval_df   = ...
pref_train_df = ...
pref_eval_df  = ...

# Convert to HuggingFace Dataset
sft_train_hf  = Dataset.from_pandas(...)
sft_eval_hf   = Dataset.from_pandas(...)
pref_train_hf = Dataset.from_pandas(...)
pref_eval_hf  = Dataset.from_pandas(...)

print(f" SFT        — Train: {len(sft_train_hf)}  Eval: {len(sft_eval_hf)}")
print(f" Preference — Train: {len(pref_train_hf)}  Eval: {len(pref_eval_hf)}")

## Cell 4 — Base Model Setup

Load the pretrained base model and snapshot its weights as `BASE_MODEL_STATE`. Define `make_fresh_causal_lm()` to create a clean copy — reused at every training stage (SFT, RM, DPO, PPO).

In [ ]:
# Load base model with correct dtype and move to DEVICE
_loader = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = ...,
    ...
).to(DEVICE)

# Save a CPU copy of all weights — used as a snapshot for later stages
BASE_MODEL_STATE = {k: v.cpu().clone() for k, v in _loader.state_dict().items()}

print(f" Total parameters : {sum(p.numel() for p in _loader.parameters())/1e6:.1f}M")


def make_fresh_causal_lm(trainable=True):
    # Load a fresh copy of the base model
    # If trainable=False: set to eval mode and freeze all parameters
    # YOUR CODE HERE
    pass

## Cell 5 — Supervised Fine-Tuning (SFT)

Fine-tune the base model on **chosen** prompt-completion pairs using `SFTTrainer` (TRL). After training, save weights to `SFT_STATE` (in memory) and `./qwen_sft_checkpoint` (on disk) — this checkpoint is the starting point for all downstream alignment stages (RM, DPO, PPO).

In [ ]:
from trl import SFTTrainer, SFTConfig

# Create a fresh trainable model
sft_model = make_fresh_causal_lm(trainable=True)

# Set gradient accumulation based on device type
grad_acc = ...

# Configure SFTConfig — set steps, batch size, lr, eval strategy,
# mixed precision (fp16/bf16), max_seq_length, dataset_text_field
sft_config = SFTConfig(
    output_dir                  = "./qwen_sft_output",
    max_steps                   = STEPS_SFT,
    per_device_train_batch_size = ...,
    gradient_accumulation_steps = ...,
    learning_rate               = ...,
    eval_strategy               = "steps",
    eval_steps                  = 10,
    use_cpu                     = ...,
    no_cuda                     = ...,
    max_seq_length              = ...,
    dataset_text_field          = ...,   # which column to train on
    # YOUR CODE HERE
)

# Build SFTTrainer with model, config, train/eval datasets, tokenizer
sft_trainer = SFTTrainer(
    model            = ...,
    args             = ...,
    train_dataset    = ...,
    eval_dataset     = ...,
    processing_class = ...,
)

sft_result = sft_trainer.train()

# Extract train and eval loss history from trainer state
sft_losses      = [(h["step"], h["loss"])     for h in sft_trainer.state.log_history if ...]
sft_eval_losses = [(h["step"], h["eval_loss"]) for h in sft_trainer.state.log_history if ...]

# Save weights in memory (used later by DPO & PPO as reference)
SFT_STATE = {k: v.cpu().clone() for k, v in sft_model.state_dict().items()}

# Save checkpoint to disk
save_dir = "./qwen_sft_checkpoint"
sft_model.save_pretrained(...)
tokenizer.save_pretrained(...)

## Cell 6 — SFT Results: Visualization & Sample Generation

Plot 3 charts from SFT training: **loss curve**, **smoothed moving average**, and **train vs eval loss** bar chart. Then generate a sample response from `sft_model` to qualitatively inspect output quality before alignment.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Plot 1 — Train and Eval loss curves over steps
if sft_losses:
    # YOUR CODE HERE
    pass

# Plot 2 — Raw loss + smoothed moving average (use np.convolve)
if sft_losses and len(sft_losses) > 3:
    w  = max(3, len(l_arr) // 5)
    sm = np.convolve(...)
    # YOUR CODE HERE
    pass

# Plot 3 — Bar chart comparing final train vs eval loss
if sft_losses and sft_eval_losses:
    # YOUR CODE HERE
    pass

plt.tight_layout()
plt.savefig("sft_results.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Sample generation ─────────────────────────────────────
sample_q  = pref_eval_df.iloc[0]["prompt"][:150]
prompt_tx = make_chatml_prompt_only(sample_q)

# Tokenize — remember: padding_side must be "left" for generation
tokenizer.padding_side = ...
enc = tokenizer(prompt_tx, return_tensors="pt", max_length=100, truncation=True).to(DEVICE)
tokenizer.padding_side = ...

# Generate response — use do_sample=True, set temperature and top_p
sft_model.eval()
with torch.no_grad():
    gen = sft_model.generate(
        **enc,
        max_new_tokens = 50,
        # YOUR CODE HERE
    )

# Decode only the newly generated tokens (skip the prompt)
resp = tokenizer.decode(gen[0][...], skip_special_tokens=True)
print(f"Prompt  : {sample_q[:100]}...")
print(f"Response: {resp[:250]}")

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 7 — Reward Model Training (Bradley-Terry)

Train a pairwise Reward Model using **Bradley-Terry loss**: `loss = -log σ(r_chosen − r_rejected)`. The model learns to score chosen responses higher than rejected ones.

Key implementation details:
- **Dynamic padding** per batch (no fixed-length waste)
- **L2 regularization** on reward scores to prevent saturation
- **Gradient accumulation** for stable updates on CPU

In [ ]:
# Fix truncation side and set a distinct pad token (not eos)
tokenizer.truncation_side = "left"
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.pad_token    = ...
    tokenizer.pad_token_id = ...

class PairwiseRewardDataset(TorchDataset):
    def __init__(self, df, tok, max_len):
        # Tokenize chosen and rejected texts — truncation=True, no padding here
        self.c_enc = tok(...)
        self.r_enc = tok(...)

    def __len__(self): ...

    def __getitem__(self, idx):
        # Return c_ids, c_mask, r_ids, r_mask as tensors
        # YOUR CODE HERE
        pass

def dynamic_collate_fn(batch):
    # Pad sequences dynamically to the longest in the batch
    # Use pad_sequence for ids (pad=pad_token_id) and masks (pad=0)
    # YOUR CODE HERE
    pass

def bradley_terry_loss(r_chosen, r_rejected, alpha=0.001):
    # BT loss: -log σ(r_chosen - r_rejected)
    # Add L2 regularization: alpha * mean(r_c² + r_r²)
    # YOUR CODE HERE
    pass

# Load reward model — AutoModelForSequenceClassification, num_labels=1
reward_model = AutoModelForSequenceClassification.from_pretrained(...)
reward_model.config.pad_token_id = tokenizer.pad_token_id

# Initialize score head weights near zero to prevent early saturation
if hasattr(reward_model, 'score'):
    reward_model.score.weight.data *= ...
    # YOUR CODE HERE

# Build DataLoaders with dynamic_collate_fn
rm_train_ldr = DataLoader(rm_train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=...)
rm_eval_ldr  = DataLoader(rm_eval_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=...)

rm_optimizer = AdamW(...)
rm_scheduler = get_linear_schedule_with_warmup(...)

# ── Training loop ─────────────────────────────────────────
ACCUM_STEPS = 4
use_scaler = False
scaler = None


reward_model.train()
rm_optimizer.zero_grad()

for step in tqdm(range(STEPS_RM), desc="RM Training"):
    batch = next(train_iter)
    for k in batch: batch[k] = batch[k].to(DEVICE)

    # Forward pass inside autocast — compute r_c and r_r, then BT loss
    # Divide loss by ACCUM_STEPS before backward
    with torch.autocast(device_type="cpu", enabled=False):
        r_c  = reward_model(...).logits.squeeze(-1)
        r_r  = reward_model(...).logits.squeeze(-1)
        loss = bradley_terry_loss(...) / ACCUM_STEPS

    # Backward — use scaler if fp16, else plain backward
    # YOUR CODE HERE

    # Optimizer step every ACCUM_STEPS — clip gradients, step, zero_grad
    if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == STEPS_RM:
        # YOUR CODE HERE
        pass

    # Every 10 steps: eval loop — compute eval loss and pairwise accuracy
    if (step + 1) % 10 == 0:
        reward_model.eval()
        with torch.no_grad():
            # YOUR CODE HERE
            pass
        reward_model.train()

## Cell 8 — Reward Model Results & get_reward()

Plot 3 charts: **BT loss curve**, **pairwise accuracy** over training, and **reward score distribution** (chosen vs rejected) on the eval set. Then define `get_reward()` — the reward signal for PPO — which returns a raw logit score for any ChatML-formatted text.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Plot 1 — BT training loss + eval loss scatter
axes[0].plot(rm_losses, ...)
# YOUR CODE HERE

# Plot 2 — Pairwise accuracy (raw + smoothed) with random baseline at 0.5
w_rm = max(3, len(rm_accuracies) // 5)
sm   = np.convolve(rm_accuracies, ...)
# YOUR CODE HERE

# Plot 3 — Histogram of chosen vs rejected reward scores on 30 eval samples
eval_c_rewards, eval_r_rewards = [], []
reward_model.eval()
with torch.no_grad():
    for i in range(min(30, len(pref_eval_df))):
        row = pref_eval_df.iloc[i]
        # Tokenize chosen and rejected — truncation=True, no static padding
        enc_c = tokenizer(row["chosen_text"],   ...)
        enc_r = tokenizer(row["rejected_text"], ...)
        eval_c_rewards.append(reward_model(**enc_c).logits.squeeze().item())
        eval_r_rewards.append(reward_model(**enc_r).logits.squeeze().item())

# Plot histogram of both distributions + vertical mean lines
# YOUR CODE HERE

plt.tight_layout()
plt.savefig("rm_results.png", dpi=150, bbox_inches="tight")
plt.show()

# Print pairwise accuracy and mean reward margin
pa = np.mean([c > r for c, r in zip(eval_c_rewards, eval_r_rewards)])
print(f" Mean reward — Chosen  : {np.mean(eval_c_rewards):.4f}")
print(f" Mean reward — Rejected: {np.mean(eval_r_rewards):.4f}")
print(f" Pairwise Accuracy     : {pa:.2%}")

# ── get_reward(): reward signal for PPO ──────────────────
def get_reward(chatml_text):
    # Tokenize input — truncation=True, no padding needed for single sample
    # Run reward_model in no_grad — return raw logit as a Python float
    # Do NOT apply sigmoid here — PPO uses the raw score
    # YOUR CODE HERE
    pass

print(f" Demo chosen  : {get_reward(pref_eval_df.iloc[0]['chosen_text']):.4f}")
print(f" Demo rejected: {get_reward(pref_eval_df.iloc[0]['rejected_text']):.4f}")

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 9 — PPO Training

Fine-tune the SFT model using **PPO** with the Reward Model as reward signal. Key components:

- **Clipped objective** — `clip(r_t, 1−ε, 1+ε)` prevents large policy updates
- **KL penalty** — keeps policy close to the frozen SFT reference
- **EMA baseline** — reduces variance in advantage estimation

In [ ]:
# Load policy (trainable) and reference (frozen) — both from SFT checkpoint
ppo_policy = AutoModelForCausalLM.from_pretrained(...).to(DEVICE)
ppo_policy.train()

ppo_ref = AutoModelForCausalLM.from_pretrained(...).to(DEVICE)
ppo_ref.eval()
# Freeze all reference parameters
# YOUR CODE HERE

use_scaler = False
scaler_ppo = None

ppo_optimizer = AdamW(ppo_policy.parameters(), lr=LR_PPO, weight_decay=0.01)

CLIP_EPS  = 0.2   # PPO clip range
MAX_KL    = 2.0   # skip step if KL exceeds this
EMA_ALPHA = 0.05  # baseline update speed

def get_response_log_probs(model, input_ids, response_mask):
    # Compute sum of log-probs only over response tokens (shift by 1)
    # Use response_mask[:, 1:] to mask prompt and padding positions
    # YOUR CODE HERE
    pass

def compute_kl_with_grad(policy_logits, ref_logits, response_mask):
    # KL(π_θ || π_ref) averaged over response tokens
    # policy_logits has gradients; ref_logits is detached
    # YOUR CODE HERE
    pass

reward_baseline = 0.5  # EMA baseline — updated each step

tokenizer.padding_side = "left"

for step in tqdm(range(STEPS_PPO), desc="PPO Training"):

    prompt_tx  = ppo_prompts[step % len(ppo_prompts)]
    enc        = tokenizer(prompt_tx, return_tensors="pt",
                           max_length=80, truncation=True, padding=False).to(DEVICE)
    prompt_len = enc["input_ids"].shape[1]

    # ── 1) Generate response (eval mode, no_grad) ─────────
    ppo_policy.eval()
    with torch.no_grad():
        gen = ppo_policy.generate(**enc, max_new_tokens=30, do_sample=True, ...)
    resp_text = tokenizer.decode(gen[0][prompt_len:], skip_special_tokens=True)
    if not resp_text.strip():
        continue

    # Build response_mask: 1 for generated tokens, 0 for prompt
    response_mask = torch.zeros_like(gen)
    response_mask[:, prompt_len:] = 1

    # ── 2) Store old log-probs and ref logits (no_grad) ───
    with torch.no_grad():
        old_log_prob = get_response_log_probs(...)
        ref_logits   = ppo_ref(input_ids=gen, ...).logits

    ppo_policy.train()

    # ── 3) Compute reward and EMA-adjusted advantage ──────
    reward_val      = get_reward(...)
    advantage       = reward_val - reward_baseline
    reward_baseline = (1 - EMA_ALPHA) * reward_baseline + EMA_ALPHA * reward_val
    adv_t           = torch.tensor(advantage, dtype=torch.float32, device=DEVICE)

    def compute_loss():
        # PPO clipped objective: -min(ratio*A, clip(ratio,1±ε)*A)
        # Add KL penalty: KL_COEF * KL(π_θ || π_ref)
        # Return (total_loss, kl_tensor)
        # YOUR CODE HERE
        pass

    # ── 4) Hard KL check — skip step if KL > MAX_KL ──────
    with torch.no_grad():
        kl_check = compute_kl_with_grad(...).item()
    if kl_check > MAX_KL:
        # log and continue without updating
        continue

    total_loss, kl_tensor = compute_loss()
    # YOUR CODE HERE

    ppo_optimizer.zero_grad()

    # Log reward, KL, loss, total reward
    ppo_reward_hist.append(reward_val)
    ppo_kl_hist.append(kl_tensor.item())
    ppo_pg_hist.append(total_loss.item())
    ppo_total_hist.append(reward_val - KL_COEF * kl_tensor.item())

tokenizer.padding_side = "right"
ppo_policy.eval()

## Cell 10 — PPO Results: Visualization & Analysis

Plot 3 charts: **reward curve** vs baseline (0.5), **KL divergence** over training, and a **reward–KL tradeoff scatter** colored by step. Then print key stats (mean reward, % above baseline, mean KL) to evaluate whether the policy improved without drifting too far from the SFT reference.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Plot 1 — Reward and Total Reward (r - β·KL) over steps
# Shade green above baseline (0.5) and red below
axes[0].plot(ppo_reward_hist, ...)
axes[0].plot(ppo_total_hist,  ...)
axes[0].axhline(0.5, ...)
axes[0].fill_between(range(len(ppo_reward_hist)), 0.5, ppo_reward_hist,
                     where=[r > 0.5  for r in ppo_reward_hist], ...)
axes[0].fill_between(range(len(ppo_reward_hist)), 0.5, ppo_reward_hist,
                     where=[r <= 0.5 for r in ppo_reward_hist], ...)
# YOUR CODE HERE

# Plot 2 — KL divergence over steps with filled area
axes[1].plot(ppo_kl_hist, ...)
axes[1].fill_between(range(len(ppo_kl_hist)), ppo_kl_hist, ...)
# YOUR CODE HERE

# Plot 3 — Scatter: KL (x) vs Reward (y), colored by step index
sc = axes[2].scatter(ppo_kl_hist, ppo_reward_hist,
                     c=range(len(ppo_kl_hist)), cmap="RdYlGn", ...)
plt.colorbar(sc, ax=axes[2], label="Training Step")
# Add mean KL (vertical) and mean Reward (horizontal) reference lines
# YOUR CODE HERE

plt.tight_layout()
plt.savefig("ppo_results.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Summary statistics ────────────────────────────────────
print(f" Mean Reward       : {np.mean(ppo_reward_hist):.4f}")
print(f" Mean KL           : {np.mean(ppo_kl_hist):.4f}")
print(f" Mean Total Reward : {np.mean(ppo_total_hist):.4f}")

# Reward trend: compare first vs last value
print(f" Reward trend      : {'↑ improving' if ... else '↓ declining'}")

# Percentage of steps where reward exceeded 0.5 baseline
print(f" % steps above 0.5 : {np.mean(...)*100:.1f}%")

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 11 — DPO Training

Train `dpo_policy` to prefer chosen over rejected responses using a frozen `dpo_ref` — no reward model needed.

**Implement:**
- `encode_with_response_mask()` — tokenize `prompt + response`, build a binary mask that is **1 only on response tokens**
- `compute_response_logprob()` — shift logits/labels by 1, gather token log-probs, sum over response mask
- `dpo_loss_fn()` — compute `loss = -log σ(β · [(log π_θ(y_w) − log π_ref(y_w)) − (log π_θ(y_l) − log π_ref(y_l))])`; ref runs inside `no_grad()`
- **Training loop** — accumulate gradients every `GRAD_ACCUM=4` steps, then clip (`max_norm=1.0`) → optimizer → scheduler

> Watch the **margin** (`r_chosen − r_rejected`): it should grow over training.

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np
from tqdm import tqdm

GRAD_ACCUM = 4

# ── Load policy model (trainable) from SFT checkpoint ──────────────────
dpo_policy = AutoModelForCausalLM.from_pretrained(
    "./qwen_sft_checkpoint",
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE)
dpo_policy.train()

# ── Load reference model (frozen) — same checkpoint as policy ──────────
dpo_ref = AutoModelForCausalLM.from_pretrained(
    "./qwen_sft_checkpoint",
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained("./qwen_sft_checkpoint", trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Freeze reference model — it must never be updated
dpo_ref.eval()
for p in dpo_ref.parameters():
    p.requires_grad = False

# ── Optimizer: AdamW + linear warmup scheduler ─────────────────────────
dpo_optimizer = AdamW(dpo_policy.parameters(), lr=LR_DPO, weight_decay=0.01)
dpo_scheduler = get_linear_schedule_with_warmup(
    dpo_optimizer,
    num_warmup_steps=10,
    num_training_steps=STEPS_DPO // GRAD_ACCUM
)

use_scaler = False
scaler_dpo = None


def encode_with_response_mask(prompt_text, full_text, tokenizer, max_length):
    # Step 1 — Tokenize prompt alone to find where the response starts (prompt_len)
    # YOUR CODE HERE

    # Step 2 — Tokenize the full text (prompt + response), truncated to max_length
    # YOUR CODE HERE

    # Step 3 — Build response_mask: zeros tensor of shape (1, T)
    #           Set positions [prompt_len : T] to 1 (response tokens only)
    # YOUR CODE HERE

    return full_enc.input_ids, full_enc.attention_mask, response_mask


def compute_response_logprob(model, input_ids, attention_mask, response_mask):
    # Forward pass through model
    # YOUR CODE HERE

    # Shift logits left by 1 and labels right by 1 (causal LM convention)
    # shift_logits = logits[:, :-1, :],  shift_labels = input_ids[:, 1:]
    # shift_rmask  = response_mask[:, 1:] — also shift the mask
    # YOUR CODE HERE

    # Compute log_softmax → gather token-level log-probs for the actual labels
    # YOUR CODE HERE

    # Mask out prompt positions, sum log-probs over response tokens only
    # YOUR CODE HERE

    return seq_logp


def dpo_loss_fn(policy, ref,
                c_ids, c_mask, c_rmask,
                r_ids, r_mask, r_rmask,
                beta):
    # Compute policy log-probs for chosen and rejected
    # YOUR CODE HERE  (pi_c, pi_r)

    # Compute reference log-probs — must be inside torch.no_grad()
    # YOUR CODE HERE  (ref_c, ref_r)

    # Compute implicit rewards: r = beta * (log_pi - log_ref)
    # YOUR CODE HERE  (r_chosen, r_rejected)

    # DPO loss = -log_sigmoid(r_chosen - r_rejected)
    # YOUR CODE HERE  → return loss, r_chosen.mean(), r_rejected.mean(), margin.mean()
    pass


# ── Training history buffers ───────────────────────────────────────────
dpo_loss_hist, dpo_chosen_hist, dpo_rejected_hist, dpo_margin_hist = [], [], [], []

print(f"\n Training DPO ({STEPS_DPO} steps, grad_accum={GRAD_ACCUM}) ...")

dpo_optimizer.zero_grad()
accum_loss = accum_margin = accum_cr = accum_rr = 0.0
update_count = 0

for step in tqdm(range(STEPS_DPO), desc="DPO Training"):
    # Sample a random row from pref_train_df
    indices = list(range(len(pref_train_df)))
    random.shuffle(indices)
    idx = indices[step % len(indices)]
    row = pref_train_df.iloc[idx]
    prompt_text = row.get("prompt_text", "")

    # Encode chosen and rejected — move all tensors to DEVICE
    c_ids, c_mask, c_rmask = encode_with_response_mask(prompt_text, row["chosen_text"],   tokenizer, MAX_LENGTH)
    r_ids, r_mask, r_rmask = encode_with_response_mask(prompt_text, row["rejected_text"], tokenizer, MAX_LENGTH)
    c_ids, c_mask, c_rmask = c_ids.to(DEVICE), c_mask.to(DEVICE), c_rmask.to(DEVICE)
    r_ids, r_mask, r_rmask = r_ids.to(DEVICE), r_mask.to(DEVICE), r_rmask.to(DEVICE)

    if USE_AMP:
        with torch.cuda.amp.autocast(dtype=DTYPE):
            # Call dpo_loss_fn and unpack (loss, cr, rr, margin)
            # YOUR CODE HERE
            pass
        # Scale loss by 1/GRAD_ACCUM before backward (gradient accumulation)
        # YOUR CODE HERE
    else:
        # YOUR CODE HERE — same without AMP
        pass

    accum_loss += loss.item();  accum_cr += cr.item()
    accum_rr   += rr.item();   accum_margin += margin.item()

    # Every GRAD_ACCUM steps: clip grads, optimizer step, scheduler step, zero_grad
    if (step + 1) % GRAD_ACCUM == 0:
        # YOUR CODE HERE — handle both scaler (AMP) and plain path
        # clip_grad_norm_ → max_norm=1.0
        # YOUR CODE HERE

        dpo_scheduler.step()
        dpo_optimizer.zero_grad()
        update_count += 1

        avg_loss   = accum_loss   / GRAD_ACCUM
        avg_cr     = accum_cr     / GRAD_ACCUM
        avg_rr     = accum_rr     / GRAD_ACCUM
        avg_margin = accum_margin / GRAD_ACCUM

        dpo_loss_hist.append(avg_loss);     dpo_chosen_hist.append(avg_cr)
        dpo_rejected_hist.append(avg_rr);   dpo_margin_hist.append(avg_margin)
        accum_loss = accum_cr = accum_rr = accum_margin = 0.0

        if update_count % (15 // GRAD_ACCUM + 1) == 0:
            print(
                f"   Update {update_count:>3} (step {step+1:>4}) | "
                f"Loss={avg_loss:.4f} | r(chosen)={avg_cr:.4f} | "
                f"r(rejected)={avg_rr:.4f} | margin={avg_margin:.4f}"
            )

dpo_policy.eval()

print(f"\n DPO complete!")
print(f" Final margin       : {dpo_margin_hist[-1]:.4f}")
print(f" Mean margin        : {np.mean(dpo_margin_hist):.4f}")
print(f" Mean margin (last 20): {np.mean(dpo_margin_hist[-20:]):.4f}")
print(f" % margin > 0       : {np.mean(np.array(dpo_margin_hist) > 0)*100:.1f}%")
print(f" Mean DPO loss      : {np.mean(dpo_loss_hist):.4f}")

## Cell 12 — DPO Results: Visualization

Plot 3 charts from `dpo_loss_hist`, `dpo_chosen_hist`, `dpo_rejected_hist`, `dpo_margin_hist`:
- **Loss curve** with smoothed overlay
- **Implicit rewards** `β·(log π_θ − log π_ref)` for chosen vs rejected — shade green where chosen > rejected
- **Margin** `r(chosen) − r(rejected)` — shade green where margin > 0

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("DPO Training Results — Qwen2.5-0.5B-Instruct", fontsize=14, fontweight="bold")

# Plot 1 — DPO loss + smoothed overlay + fill_between
w_d = max(3, len(dpo_loss_hist) // 6)
sm  = np.convolve(dpo_loss_hist, np.ones(w_d) / w_d, mode="valid")
# YOUR CODE HERE

# Plot 2 — Implicit rewards: chosen vs rejected lines
#           fill_between green where chosen > rejected, red where chosen <= rejected
#           add axhline at y=0 (gray dashed)
# YOUR CODE HERE

# Plot 3 — Margin curve
#           fill_between green where margin > 0, red where margin <= 0
#           add axhline at y=0 (black dashed)
# YOUR CODE HERE

plt.tight_layout()
plt.savefig("dpo_results.png", dpi=150, bbox_inches="tight")
plt.show()

# Print: % margin > 0, mean margin (all), mean margin (last 20), mean loss (last 20)
# YOUR CODE HERE

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 13 — ORPO Training

Train a **single model** with no reference copy. ORPO combines SFT and preference alignment in one loss:

$$\mathcal{L}_{\text{ORPO}} = \mathcal{L}_{\text{SFT}} + \lambda \cdot (-\log \sigma(\text{log-odds}(y_w) - \text{log-odds}(y_l)))$$

**Implement:**
- `compute_log_prob_response()` — same as DPO cell but **divide by response length** (per-token average)
- `compute_orpo_loss()` — compute `l_sft` (cross-entropy on chosen tokens), then `safe_log_odds()` to convert avg log-prob → log-odds, then `l_or = -log σ(log_odds_chosen − log_odds_rejected)`; return `l_sft + λ·l_or`
- **Training loop** — no gradient accumulation; `zero_grad → backward → clip → step` every step

In [ ]:
orpo_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True,
).to(DEVICE)
orpo_model.train()

orpo_optimizer = AdamW(orpo_model.parameters(), lr=LR_ORPO, weight_decay=0.01)
orpo_scheduler = get_linear_schedule_with_warmup(
    orpo_optimizer, num_warmup_steps=5, num_training_steps=STEPS_ORPO
)

# encode_with_response_mask — reuse same function from DPO cell (prompt_len cap: max_length // 2)

def compute_log_prob_response(model, input_ids, attention_mask, response_mask):
    # Forward pass, shift logits/labels/mask by 1
    # Gather token log-probs, mask prompt tokens
    # Return MEAN log-prob over response tokens (divide by n_resp)
    # YOUR CODE HERE
    pass

def compute_orpo_loss(model, c_ids, c_mask, c_rmask, r_ids, r_mask, r_rmask, lam):
    # ── SFT loss: cross-entropy on chosen response tokens only ──────────
    # Forward pass on chosen, compute per-token CE with reduction='none'
    # Mask prompt tokens, average over response length → l_sft
    # YOUR CODE HERE

    # ── Odds ratio loss ─────────────────────────────────────────────────
    # Get avg log-probs for chosen and rejected via compute_log_prob_response
    # YOUR CODE HERE  (lp_c, lp_r)

    def safe_log_odds(log_p):
        # Clamp log_p to (-10, -1e-4), then return log(p / (1 - p + 1e-8))
        # YOUR CODE HERE
        pass

    # log_odds_ratio = safe_log_odds(lp_c) - safe_log_odds(lp_r)
    # l_or = -log_sigmoid(log_odds_ratio).mean()
    # l_total = l_sft + lam * l_or
    # YOUR CODE HERE
    # return l_total, l_sft.item(), l_or.item(), log_odds_ratio.mean().item()
    pass

orpo_loss_hist, orpo_sft_hist, orpo_or_hist, orpo_lor_hist = [], [], [], []

use_scaler = False
scaler_orpo = None

print(f"\n Training ORPO ({STEPS_ORPO} steps) ...")
for step in tqdm(range(STEPS_ORPO), desc="ORPO Training"):
    row = pref_train_df.iloc[step % len(pref_train_df)]
    prompt_text = row.get("prompt_text", "")

    # Encode chosen and rejected — move to DEVICE
    # YOUR CODE HERE

    if USE_AMP:
        with torch.cuda.amp.autocast(dtype=DTYPE):
            # Call compute_orpo_loss, unpack 4 values
            # YOUR CODE HERE
            pass
    else:
        # YOUR CODE HERE
        pass

    # zero_grad → backward (with scaler if AMP) → clip max_norm=1.0 → step → scheduler
    # YOUR CODE HERE

    orpo_loss_hist.append(loss.item())
    orpo_sft_hist.append(l_sft)
    orpo_or_hist.append(l_or)
    orpo_lor_hist.append(lor)

    if (step + 1) % 15 == 0:
        print(f"   Step {step+1:>3} | Total={loss.item():.4f} | "
              f"SFT={l_sft:.4f} | OR={l_or:.4f} | log_ratio={lor:.4f}")

orpo_model.eval()
print(f"\n ORPO complete! % log_ratio > 0: {np.mean(np.array(orpo_lor_hist)>0)*100:.1f}%")

## Cell 14 — ORPO Results: Visualization

Plot 3 charts from `orpo_loss_hist`, `orpo_sft_hist`, `orpo_or_hist`, `orpo_lor_hist`:
- **Loss decomposition** — total, `L_SFT`, and `λ·L_OR` on the same axes
- **Log odds ratio** with smoothed overlay — shade green where > 0, red where ≤ 0
- **Scatter** `L_SFT` vs `L_OR` colored by training step (viridis colormap)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("ORPO Training Results — Qwen2.5-0.5B-Instruct", fontsize=14, fontweight="bold")

# Plot 1 — 3 lines on same axes: total loss, l_sft (dashed), λ·l_or (dotted)
# YOUR CODE HERE

# Plot 2 — Log odds ratio raw + smoothed overlay + axhline at 0
#           fill_between green where > 0, red where <= 0
w_o  = max(3, len(orpo_lor_hist) // 6)
sm_o = np.convolve(orpo_lor_hist, np.ones(w_o) / w_o, mode="valid")
# YOUR CODE HERE

# Plot 3 — Scatter(sft_hist, or_hist) colored by step index with viridis cmap
#           add colorbar with label "Training Step"
# YOUR CODE HERE

plt.tight_layout()
plt.savefig("orpo_results.png", dpi=150, bbox_inches="tight")
plt.show()

# Print: % log_ratio > 0, mean OR/SFT/Total loss (last 20 steps)
# YOUR CODE HERE

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 15 — Final Comparison: All Methods

Build a 3×3 dashboard comparing SFT, RM, PPO, DPO, and ORPO across 5 panels:
- **Loss curves** (normalized x-axis) + **bar chart** of final losses
- **PPO reward**, **DPO margin**, **ORPO log-odds ratio** — each with green/red `fill_between`
- **Radar chart** scoring all methods on 5 qualitative axes (Simplicity, Efficiency, Stability, Quality, No-Ref)

In [ ]:
fig = plt.figure(figsize=(22, 15))
fig.suptitle(
    "Comprehensive Alignment Method Comparison — Qwen2.5-0.5B-Instruct on Anthropic HH-RLHF",
    fontsize=17, fontweight="bold", y=0.99
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38)

# ── ax1 (gs[0, :2]) — All training loss curves on a normalised x-axis [0,1] ──
ax1 = fig.add_subplot(gs[0, :2])
# Plot each available history (check with `if hist:`) using np.linspace(0,1,len(hist))
# YOUR CODE HERE

# ── ax2 (gs[0, 2]) — Bar chart: final loss per method (mean of last 10 steps) ──
ax2 = fig.add_subplot(gs[0, 2])
fl = {
    "SFT":    sft_losses[-1][1]            if sft_losses    else 0,
    "RM":     rm_losses[-1]                if rm_losses     else 0,
    "PPO\n(PG)": abs(np.mean(ppo_pg_hist[-5:])) if ppo_pg_hist else 0,
    "DPO":    np.mean(dpo_loss_hist[-10:]) if dpo_loss_hist else 0,
    "ORPO":   np.mean(orpo_loss_hist[-10:])if orpo_loss_hist else 0,
}
# Plot bars with COLORS, add value label above each bar (fontsize=9, bold)
# YOUR CODE HERE

# ── ax3 (gs[1, 0]) — PPO reward curve + fill_between vs baseline 0.5 ─────────
ax3 = fig.add_subplot(gs[1, 0])
# YOUR CODE HERE

# ── ax4 (gs[1, 1]) — DPO margin + fill_between vs 0 ──────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
# YOUR CODE HERE

# ── ax5 (gs[1, 2]) — ORPO log-odds ratio + fill_between vs 0 ─────────────────
ax5 = fig.add_subplot(gs[1, 2])
# YOUR CODE HERE

# ── ax6 (gs[2, 0]) — Radar chart (polar) ─────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0], polar=True)
cats = ["Simplicity", "Resource\nEfficiency", "Training\nStability",
        "Final\nQuality", "No Ref\nModel"]
scores = {
    "SFT":  [5, 5, 5, 2, 5],
    "PPO":  [1, 1, 2, 5, 1],
    "DPO":  [3, 3, 4, 4, 3],
    "ORPO": [4, 4, 4, 4, 5],
}
# Compute angles: N evenly-spaced points on [0, 2π], close the loop by appending angs[0]
# Set xticks/yticks, plot each method with fill(alpha=0.07), add legend
# YOUR CODE HERE

plt.savefig("final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 16 — Qualitative Generation Comparison

Generate responses to the same prompt from SFT, DPO, and ORPO models side-by-side.

**Implement** `generate()` — format the prompt with `make_chatml_prompt_only()`, tokenize with `padding_side="left"`, then call `model.generate()` with sampling (`do_sample=True`, `temperature`, `top_p`) and decode only the **new tokens** (slice from `input_ids.shape[1]:`).

In [ ]:
sample_q = pref_eval_df.iloc[2]["prompt"][:400]
print(f" Prompt: {sample_q[:1000]}...\n")
print("─" * 70)

def generate(model, question, max_new=80, temp=0.7):
    # Format prompt using make_chatml_prompt_only
    # YOUR CODE HERE

    # Tokenize with padding_side="left", max_length=100, truncation=True → move to DEVICE
    # YOUR CODE HERE

    model.eval()
    with torch.no_grad():
        # Call model.generate() — do_sample=True, temperature, top_p=0.9
        # pad_token_id = eos_token_id
        # YOUR CODE HERE
        pass

    # Decode only new tokens (skip input_ids length), skip_special_tokens=True
    # YOUR CODE HERE

models_cmp = [
    ("🔵 SFT",  sft_model,  "Supervised Fine-Tuning on chosen (CE loss)"),
    ("🟣 DPO",  dpo_policy, "Direct Preference Optimisation (log-ratio)"),
    ("🟢 ORPO", orpo_model, "SFT + Odds Ratio, no reference model"),
]

for name, mdl, desc in models_cmp:
    print(f"\n{name} — {desc}:")
    resp = generate(mdl, sample_q)
    print(f"  {resp[:300]}")
    print("─" * 70)

## LLM As a Judge 

## Cell 17 - Data Loading

The **MT-Bench Human Judgments** dataset contains pairwise comparisons of responses
from different LLMs, each labeled with a human preference judgment.

Each row contains two conversations (`conversation_a`, `conversation_b`) with this structure:
[ {role: "user", content: "question"}, {role: "assistant", content: "answer"} ]

**Task:** Implement the `extract(row)` function that pulls out the question
and both model answers from a single dataset row.

In [ ]:
print("Loading MT-Bench Human Judgments...")
df = load_dataset("lmsys/mt_bench_human_judgments", split="human").to_pandas()

print(f" Samples: {len(df):,}  |  Columns: {list(df.columns)}")
print(f"\n Human Judgment Distribution:")
print(df['winner'].value_counts().to_string())

def extract(row):
    # TODO: extract the question from conversation_a (index 0, key 'content')
    question = ...
    # TODO: extract model A's answer from conversation_a (index 1, key 'content')
    answer_a = ...
    # TODO: extract model B's answer from conversation_b (index 1, key 'content')
    answer_b = ...
    return question, answer_a, answer_b

s = df.iloc[3]
q, a, b = extract(s)

SEP = "─" * 65
print(f"\n{SEP}\n Question:\n{q[:250]}...")
print(f"{SEP}\n Answer A [{s['model_a']}]:\n{a[:200]}...")
print(f"{SEP}\n Answer B [{s['model_b']}]:\n{b[:200]}...")
print(f"{SEP}\n Human Judgment: {s['winner']}")

##  Cell 18 - Loading the Judge Model

We use **`google/flan-t5-base`** (250M parameters) as our judge — a small
instruction-tuned T5 model that runs entirely on CPU.

Two components need to be loaded:
- **Tokenizer** — converts text to token IDs and back
- **Model** — the T5 weights that generate the verdict

After loading, always set the model to **eval mode** to disable dropout
and ensure deterministic outputs during inference.

In [ ]:
MODEL_NAME = "google/flan-t5-base"
print(f" Loading {MODEL_NAME} ...")

# TODO: load the tokenizer using T5Tokenizer and MODEL_NAME
tokenizer = ...

# TODO: load the model using T5ForConditionalGeneration and MODEL_NAME
model = ...

# TODO: set the model to evaluation mode
...

print(f" Model is ready! Parameters: {sum(p.numel() for p in model.parameters()):,}")

##  Cell 19 - Judge Prompt & Inference Pipeline

The core of LLM-as-a-Judge is a **pairwise comparison prompt** that instructs
the model to compare two answers and return a single verdict: `A`, `B`, or `tie`.

The inference pipeline has three steps:
1. **Tokenize** — encode the prompt into token IDs (`return_tensors="pt"`, `truncation=True`)
2. **Generate** — run the model with `num_beams=2` for slightly more stable outputs
3. **Parse** — decode the output and map raw text → `'model_a'` / `'model_b'` / `'tie'`

**Task:** Write the judge prompt and complete the three inference steps.

In [ ]:
# TODO: write a pairwise judge prompt with three placeholders: {q}, {a}, {b}
# The prompt should instruct the model to reply with exactly one word: A, B, or tie
PROMPT = """..."""


def llm_judge(question: str, ans_a: str, ans_b: str) -> str:
    prompt = PROMPT.format(
        q=question[:300],
        a=ans_a[:250],
        b=ans_b[:250]
    )

    # TODO: tokenize the prompt using the tokenizer
    # (return_tensors="pt", max_length=512, truncation=True)
    inputs = ...

    # TODO: generate output tokens using model.generate
    # (max_new_tokens=8, num_beams=2, early_stopping=True)
    outputs = ...

    # TODO: decode outputs[0] with the tokenizer, strip and lowercase the result
    raw = ...

    # TODO: parse raw into 'model_a', 'model_b', or 'tie'
    # hint: check if raw starts with 'a' or 'b', or contains 'tie'
    if ...:
        return 'model_a'
    elif ...:
        return 'model_b'
    elif ...:
        return 'tie'
    elif ...:
        return 'model_a'
    elif ...:
        return 'model_b'
    else:
        return 'tie'


q0, a0, b0 = extract(df.iloc[3])
verdict = llm_judge(q0, a0, b0)

print(f" Test: Model Verdict = [{verdict}]  |  Human Judgment = [{df.iloc[3]['winner']}]")

##  Cell 20 - Running the Evaluation

We run `llm_judge()` on **10 samples** (turn=1 only) and compare each verdict
against the human label to measure **Agreement Rate**.

One important detail: human labels contain variants like `"tie (bothbad)"`,
so they must be **normalized** before comparison:
- `"tie"` or `"tie (bothbad)"` → `"tie"`
- `"model_a"` / `"model_b"` → unchanged

**Task:** Fill in the normalization logic, the match check, and the results dictionary.

In [ ]:
sample = (
    df[df['turn'] == 1]
    .sample(10, random_state=7)
    .reset_index(drop=True)
)

records = []
SEP = "─" * 68

print(f"{'#':<3} {'Model A':<17} {'Model B':<17} {'Judge':<10} {'Human':<17} {'✓'}")
print(SEP)

for _, row in sample.iterrows():
    q, a, b = extract(row)
    verdict = llm_judge(q, a, b)
    human = row['winner']

    # TODO: normalize human label — map any string containing 'tie' to 'tie'
    hnorm = ...

    # TODO: check if the judge verdict matches the normalized human label
    ok = ...

    # TODO: append a dict with keys: model_a, model_b, human, hnorm, verdict, ok
    records.append(dict(
        ...
    ))

    n = len(records)
    print(
        f"{n:<3} {row['model_a']:<17} {row['model_b']:<17} "
        f"{verdict:<10} {human:<17} {'✅' if ok else '❌'}"
    )

res = pd.DataFrame(records)
print(
    f"\n Finished!  Agreement Rate = {res['ok'].mean():.0%}  "
    f"({res['ok'].sum()}/{len(res)})"
)

## Cell 21 - Visualizing Results

Two plots side by side to analyze judge performance:

- **Left — Pie chart:** Agreement Rate between LLM Judge and human labels
- **Right — Bar chart:** Distribution of verdicts (`model_a` / `model_b` / `tie`)
  comparing human vs LLM Judge counts side by side

**Task:** Fill in the data for both plots.
For the bar chart, use a bar width of `0.35` and offset each group
by `±w/2` so the two bars sit side by side without overlapping.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("LLM as a Judge — Evaluation Results", fontsize=13, fontweight='bold')
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# ── Left: Pie chart ───────────────────────────────────────────────────────
ax = axes[0]

# TODO: calculate number of correct and total predictions from res
n_ok    = ...
n_total = ...

ax.pie(
    # TODO: pass [match_count, mismatch_count] as the pie values
    [..., ...],
    labels=[f"Match\n({n_ok})", f"Mismatch\n({n_total - n_ok})"],
    colors=['#27ae60', '#e74c3c'],
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
    textprops=dict(fontsize=11, fontweight='bold')
)
ax.set_title(f"Agreement Rate: {n_ok / n_total:.0%}", fontsize=12, fontweight='bold')

# ── Right: Grouped bar chart ──────────────────────────────────────────────
ax = axes[1]

cats = ['model_a', 'model_b', 'tie']
lbl  = ['A Better', 'B Better', 'Tie']

# TODO: for each category in cats, count how many times it appears
# in res['hnorm'] (human) and res['verdict'] (LLM judge)
hc = [... for c in cats]
lc = [... for c in cats]

x, w = range(3), 0.35

# TODO: plot human bars shifted left by w/2 and LLM Judge bars shifted right by w/2
b1 = ax.bar([... for i in x], hc, w, label='Human',     color='#3498db', alpha=0.85, edgecolor='white')
b2 = ax.bar([... for i in x], lc, w, label='LLM Judge', color='#e67e22', alpha=0.85, edgecolor='white')

for bar in list(b1) + list(b2):
    v = int(bar.get_height())
    if v:
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.05, str(v),
                ha='center', fontsize=11, fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(lbl, fontsize=10)
ax.set_ylabel('Count')
ax.legend(fontsize=10)
ax.set_title('Human vs LLM Judge', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('./llm_judge_results.png', dpi=150, bbox_inches='tight')
plt.show()

Chart Analysis
 
**a)** Analyze the results shown in the charts below.


## Cell 22 - Position Bias Test

This cell checks whether the judge changes its decision when answers A and B are swapped.

- Compare the original order `(A, B)` with the swapped order `(B, A)`
- Check whether the verdict remains consistent after swapping
- Count how many cases show position bias

**Task:** Complete the missing logic for the swapped comparison and consistency check.

In [ ]:
print(" Position Bias Test (5 Samples)\n")

SEP = "─" * 55

print(f"{'#':<3} {'Original (A,B)':<14} {'Swapped (B,A)':<14} {'Consistent?'}")
print(SEP)

bias = []

for i, row in sample.head(5).iterrows():

    q, a, b = extract(row)

    # Judge the original order
    v_orig = llm_judge(q, a, b)
    
    # Judge the swapped order
    v_swap = llm_judge(q, b, a)

    # TODO: set the expected verdict after swapping A and B
    expected = ...

    # TODO: check whether the swapped verdict matches the expected one
    ok = ...

    bias.append(ok)

    # TODO: choose the right icon
    icon = ...

    print(f"{len(bias):<3} {v_orig:<14} {v_swap:<14} {icon}")

print(SEP)

# TODO: compute the consistency rate
rate = ...

print(f"\n Consistency Rate: {rate:.0%}  |  Bias Cases: {bias.count(False)}/5")

print(
    "\n Position Bias means the model gives a different verdict "
    "for the same pair of answers only because the positions of "
    "A and B were swapped."
)

## Cell 23 - Questions

**a)** What does alignment mean in large language models, and why is training on general text data alone insufficient to achieve it?

**b)** What is the difference between Supervised Fine-Tuning (SFT) and pretraining, and why is SFT essential for alignment?

**c)** How does PPO work in RLHF, and why is it used for alignment?

**d)** What are the differences between DPO and PPO? Then explain what idea ORPO combines from both SFT and DPO.

**e)** In PPO, what happens if the KL coefficient is set too large (e.g., β = 10)? And what happens if β = 0?

**f)** How does DPO differ from RLHF in aligning large language models? Explain the DPO loss function provided below and describe the role of each of its main components.

**g)** Explain how the PPO algorithm mitigates the risk of overoptimization or instability when aligning large language models with human preferences.



## Cell 24 -Additional notes on ORPO

**ORPO** (Odds Ratio Preference Optimization) is an alignment method for large language models that, instead of using a separate RLHF stage or a reference model, integrates the SFT process itself with a preference penalty. The core idea is that the model, when trained on preference data, assigns higher probability to preferred responses and lower probability to rejected ones but achieves this through a simpler loss function that adds an odds ratio-based term to the standard cross-entropy loss.
For this reason, ORPO can be viewed as a unified training approach that simultaneously fine-tunes the model on correct-style data and discourages it from learning undesirable styles.
 
A key advantage of ORPO is its simplicity and efficiency: it requires neither a reference model, a reward model, nor a separate preference optimization stage — making it lighter in terms of both implementation and computational cost compared to more complex methods. According to the paper, this approach has been tested across various model sizes and has shown competitive or even superior results compared to some larger-scale methods.


![chart](https://media.licdn.com/dms/image/v2/D5612AQEwV9lkzYhAkQ/article-cover_image-shrink_600_2000/article-cover_image-shrink_600_2000/0/1714566247704?e=2147483647&v=beta&t=yrLsrE91mVchsO1kf96pYroZHU34MvXqNxZQEqAX61g)
 
---
 
**Paper-Link:** https://arxiv.org/abs/2403.07691
 
